### Import data

In [5]:
import pandas as pd

# Load data into pandas DataFrames
path = 'original_csv/'
brands = pd.read_csv(path+'brands.csv')
orderlines = pd.read_csv(path+'orderlines.csv')
orders = pd.read_csv(path+'orders.csv')
products = pd.read_csv(path+'products.csv')

In [6]:
# create copies of the original data
orders_df = orders.copy()
orderlines_df = orderlines.copy()
brands_df = brands.copy()
products_df = products.copy()

### Duplicates

In [7]:
# look for duplicates
print(orders_df.duplicated().sum())
print(orderlines_df.duplicated().sum())
print(products_df.duplicated().sum())
print(brands.duplicated().sum())

0
0
8746
0


In [8]:
# delete duplicates in products
products_df = products_df.drop_duplicates()

In [9]:
# look for duplicates in primary keys and other important columns
print(orders_df.duplicated('order_id').sum())
print(orderlines_df.duplicated('id').sum())
print(products_df.duplicated('sku').sum())
print(brands_df.duplicated('short').sum())
print(brands_df.duplicated('long').sum())

0
0
1
0
6


In [ ]:
# look at multiple brand duplicates in long row

brands_df.loc[brands_df["long"].duplicated(keep=False)]

,short,long
6,AP2,Apple
7,APP,Apple
17,BOS,Bose
19,CAD,Bose
37,ENV,Unknown
67,JYB,Jaybird
70,KEN,Jaybird
80,LIB,Unknown
94,MOP,Mophie
98,MUJ,Mophie


In [ ]:
# Fix KEN and MUJ (OTR and STA are complicated, so I am not touching them)
brands_df.loc[70,'long'] = 'Kensington'
brands_df.loc[98,'long'] = 'Mujjo'

# remove rows 19, 37, 80
brands_df = brands_df.drop([19,37,80])

In [11]:
# look at the product sku duplicate

products_df.loc[products_df["sku"].duplicated(keep=False)]

# its ok to leave the second instance, as this will be cleaned by deleting rows with NaN values in price later

,sku,name,desc,price,promo_price,in_stock,type
7992,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,1729,1305.59,0,1282
8000,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,NaN,1305.59,0,1282


### Missing values

In [12]:
# Orders: delete 5 missing value rows
orders_df = orders_df.dropna(axis=0)

In [13]:
# Orderlines: product_id is useless (only 1's), so delete column
orderlines_df = orderlines_df.drop(columns=["product_id"])

In [14]:
# Products: missing values: descriptions
# 7 descriptions are NaN, just fill them with the name
products_df.loc[products_df['desc'].isna(), 'desc'] = products_df.loc[products_df['desc'].isna(), 'name']

In [15]:
# Products: missing values: price
# missing prices
print(f"The missing values in price are {(products_df.price.isna().value_counts(normalize=True).iloc[1] * 100).round(2)}% of all rows in the DataFrame")

# delete 0.43% of price rows with NaN
products_df = products_df.dropna(subset=['price'])

The missing values in price are 0.43% of all rows in the DataFrame


### Data type conversions

In [16]:
#orders
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

In [17]:
#orderlines
orderlines_df['date'] = pd.to_datetime(orderlines_df['date'])

In [18]:
# orderlines: unit price has weird format, e.g. 3.222.512
orderlines_df['unit_price'].str.count(r"\.").value_counts()

# Count the rows with more than one `.`
mult_decimal_rows = (orderlines_df['unit_price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
percent_corrupted = (100 * mult_decimal_rows / orderlines_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in orderlines_df['unit_price'] have three decimals after last point")

12.30% of the rows in orderlines_df['unit_price'] have three decimals after last point


In [19]:
# Decision: remove unit_price rows with two dots
two_dot_order_ids_list = orderlines_df.loc[orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+"), "id_order"]
orderlines_df = orderlines_df.loc[~orderlines_df.id_order.isin(two_dot_order_ids_list)]
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

In [20]:
# ## ALTERNATIVE: cleaning unit_price
# # orderlines: remove first '.' in unit prices with multiple periods (have weird format, e.g. 3.222.52) and convert to float
# mult_decimal_mask = (orderlines_df['unit_price'].str.count(r"\.")>1)
# mult_decimal_orderlines_df = orderlines_df.loc[mult_decimal_mask]
# orderlines_df.loc[mult_decimal_mask,'unit_price'] = orderlines_df.loc[mult_decimal_mask,'unit_price'].str.replace('.', '', n=1)
# orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

In [21]:
# Products: Look for wrongly formated numbers in products_df['price'], e.g. 123.456.789

# mask for numbers with 3 digits after last . in price
mask_price_digits = products_df['price'].str.split('.').str[-1].str.len()==3
# mask for numbers with more than 0 x '.'
mask_price_periods = products_df['price'].str.count(r"\.")>0
# combined mask
combined_price_mask = mask_price_periods & mask_price_digits

percent_corrupted = (100 * combined_price_mask.sum() / products_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in products_df['price'] have three decimals after last point")

5.15% of the rows in products_df['price'] have three decimals after last point


In [22]:
# Products: Look for wrongly formated numbers in orderlines and products merge['price'], e.g. 123.456.789

merge = products_df.merge(orderlines_df,on='sku', how='inner')
merge_mask1 = merge['price'].str.split('.').str[-1].str.len()==3
merge_mask2 = merge['price'].str.count(r"\.")>0
combined_merge_mask = merge_mask1 & merge_mask2


percent_corrupted = (100 * combined_merge_mask.sum() / merge.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in merge['price'] have three decimals after last point")

merge.loc[combined_merge_mask,'id'].nunique()

2.51% of the rows in merge['price'] have three decimals after last point


5400

In [23]:
# decision: delete these 5% instances of double periods or 3 decimals in price rows in products
products_df = products_df.loc[~((products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}"))), :]

In [24]:
# convert price column to float
products_df["price"] = pd.to_numeric(products_df["price"])

In [25]:
# Products: promo_price
# mask for numbers with 3 digits after last . in price
mask_promo_price_digits = (products_df['promo_price'].str.split('.').str[-1].str.len()==3)
# mask for numbers with more than 0 x '.'
mask_promo_price_periods = products_df['promo_price'].str.count(r"\.")>0
# combined mask
combined_promo_price_mask = mask_promo_price_digits & mask_promo_price_periods

percent_corrupted = (100 * combined_promo_price_mask.sum() / products_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in products_df['promo_price'] have three decimals after last point")

92.39% of the rows in products_df['promo_price'] have three decimals after last point


In [26]:
# decision: drop the promo_price column
products_cl = products_df.drop(columns=["promo_price"])


In [ ]:
# # Alterantive: keep promo price with the following fix:
# # This function goes line by line in the products_df
# # products_df should be already cleaned (no NaN values and price needs to be robust)
# # It fixes promo_prices that have 3 digits after the last period (about 92% of rows)
# # It leaves out promo_prices with 1-2 digits after the last period, as these seem correct
# # It leaves out promo_prices with no periods, as these seem correct
# # There are correct promo_prices with 3 digits after the last period
# # If the promo_price deviates too much from the price (over 9x), it is handled differently

# products_df_test = products_df.copy()

# def fix_price_with_reference(row):
#     pp = row['promo_price']
#     p = row['price']

#     if pp.count('.') == 0:
#         return pp
#     if (len(pp.split('.')[-1]) == 1) or (len(pp.split('.')[-1]) == 2):
#         return pp

#     raw = pp.replace('.', '')
#     candidate_fixed = round(float(raw) / 10000, 2)

#     if p / candidate_fixed > 9:
#         return str(round(float(raw) / 1000, 2))
#     else:
#         return str(candidate_fixed)

# # apply function to all rows in products_df
# products_df_test.loc[:,'promo_price'] = products_df_test.apply(fix_price_with_reference, axis=1)

# #change data type of promo_price to float
# products_df_test['promo_price'] = products_df_test['promo_price'].astype(float)

Check dataframes before saving

In [28]:
products_cl.info()

<class 'pandas.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   str    
 1   name      9992 non-null   str    
 2   desc      9992 non-null   str    
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 546.4 KB


In [29]:
orderlines_df.info()

<class 'pandas.DataFrame'>
Index: 216250 entries, 0 to 293982
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                216250 non-null  int64         
 1   id_order          216250 non-null  int64         
 2   product_quantity  216250 non-null  int64         
 3   sku               216250 non-null  str           
 4   unit_price        216250 non-null  float64       
 5   date              216250 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3), str(1)
memory usage: 11.5 MB


In [30]:
orders_df.info()


<class 'pandas.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[us]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 8.7 MB


Save dataframes

In [31]:
save_path = 'cleaned_csv/'
orders_df.to_csv(save_path+'orders_cl.csv', index=False)
orderlines_df.to_csv(save_path+'orderlines_cl.csv', index=False)
products_cl.to_csv(save_path+'products_cl.csv', index=False)
brands_df.to_csv(save_path+'brands_cl.csv', index=False)